# 线性代数直觉（Linear Algebra Intuition）

对应课程：`phases/01-math-foundations/01-linear-algebra-intuition`

> 每个 AI 模型都只是戴着花哨帽子的矩阵数学。

本 notebook 把 `vectors.py` 里的核心函数拆开：每个函数一组中文注释，后面跟一小段可运行实验。完整打印型 demo 仍在 `vectors.py`。

**贯穿全课的模式：** 向量是空间里的箭头；点积量相似，投影拆方向，Gram-Schmidt 换成互相垂直的尺子。


## 学习目标（Learning Objectives）

- 从零实现向量/矩阵运算（加法、点积、矩阵乘）
- 几何上解释点积、投影、Gram-Schmidt
- 用行化简判断线性无关、秩、基
- 连到 embedding、注意力分数、LoRA


## 0. 依赖

只用标准库，和课程允许清单一致。


In [1]:
import math


## 1. Vector：加法、减法、标量乘、点积

向量是一列数字，也是从原点出发的箭头。逐元素加减平移箭头；标量乘拉长或缩短；点积把两支箭头收成一个数：

$$
\mathbf{a}\cdot\mathbf{b} = \sum_i a_i b_i
$$

同向为正、正交为 0、反向为负。嵌入相似度、注意力分数，底层都是这个和。


In [2]:
class Vector:
    def __init__(self, components):
        """一列数字。dim 记长度，后面的运算都按分量走。"""
        self.components = list(components)
        self.dim = len(self.components)

    def __add__(self, other):
        """逐元素相加。zip 以较短者为准。"""
        return Vector([a + b for a, b in zip(self.components, other.components)])

    def __sub__(self, other):
        """逐元素相减。"""
        return Vector([a - b for a, b in zip(self.components, other.components)])

    def __mul__(self, scalar):
        """标量乘法：每个分量乘同一个数。"""
        return Vector([x * scalar for x in self.components])

    def dot(self, other):
        """点积：对应分量相乘再求和。"""
        return sum(a * b for a, b in zip(self.components, other.components))

    def __repr__(self):
        return f"Vector({self.components})"


a = Vector([1, 2, 3])
b = Vector([4, 5, 6])
print("a =", a)
print("b =", b)
print("a + b =", a + b)
print("a - b =", a - b)
print("a * 3 =", a * 3)
print("a · b =", a.dot(b))


a = Vector([1, 2, 3])
b = Vector([4, 5, 6])
a + b = Vector([5, 7, 9])
a - b = Vector([-3, -3, -3])
a * 3 = Vector([3, 6, 9])
a · b = 32


## 2. 模长与归一化

$$
\|\mathbf{v}\| = \sqrt{\mathbf{v}\cdot\mathbf{v}} = \sqrt{\sum_i v_i^2}
$$

归一化 $\hat{\mathbf{v}} = \mathbf{v}/\|\mathbf{v}\|$ 只保留方向。`[3, 4]` 是 3-4-5 直角三角形，模长必须是 5。零向量没有方向，除零会炸。


In [3]:
def magnitude(self):
    """欧几里得长度。"""
    return sum(x ** 2 for x in self.components) ** 0.5


def normalize(self):
    """单位向量。零向量会 ZeroDivisionError。"""
    mag = self.magnitude()
    return Vector([x / mag for x in self.components])


Vector.magnitude = magnitude
Vector.normalize = normalize

v = Vector([3, 4])
print("| [3, 4] | =", v.magnitude())
n = v.normalize()
print("归一化 =", n)
print("|归一化| =", n.magnitude())


| [3, 4] | = 5.0
归一化 = Vector([0.6, 0.8])
|归一化| = 1.0


## 3. 余弦相似度

$$
\cos\theta = \frac{\mathbf{a}\cdot\mathbf{b}}{\|\mathbf{a}\|\,\|\mathbf{b}\|}
$$

只看夹角、不看长度。正交向量点积为 0，余弦也是 0——词嵌入里这叫「无关」。


In [4]:
def cosine_similarity(self, other):
    """cos θ。任一向量为零会除零。"""
    return self.dot(other) / (self.magnitude() * other.magnitude())


Vector.cosine_similarity = cosine_similarity

ex = Vector([1, 0])
ey = Vector([0, 1])
print("正交 [1,0] 与 [0,1] 的余弦 =", ex.cosine_similarity(ey))
print("同向 [1,0] 与 [2,0] 的余弦 =", ex.cosine_similarity(Vector([2, 0])))
print("反向 [1,0] 与 [-1,0] 的余弦 =", ex.cosine_similarity(Vector([-1, 0])))


正交 [1,0] 与 [0,1] 的余弦 = 0.0
同向 [1,0] 与 [2,0] 的余弦 = 1.0
反向 [1,0] 与 [-1,0] 的余弦 = -1.0


## 4. 夹角

$$
\theta = \arccos(\mathrm{clip}(\cos\theta,\,-1,\,1))
$$

浮点误差可能让余弦略超出 $[-1,1]$，`acos` 会变成 NaN，所以先夹紧。返回度数，方便对照「直角 = 90°」。


In [5]:
def angle_between(self, other):
    """两向量夹角，单位：度。"""
    cos_theta = self.cosine_similarity(other)
    cos_theta = max(-1.0, min(1.0, cos_theta))
    return math.degrees(math.acos(cos_theta))


Vector.angle_between = angle_between

v1 = Vector([1, 0])
v2 = Vector([0, 1])
v3 = Vector([1, 1])
print("夹角 [1,0] 与 [0,1]:", v1.angle_between(v2), "度")
print("夹角 [1,0] 与 [1,1]:", round(v1.angle_between(v3), 1), "度")
print("夹角 [1,0] 与自己:", v1.angle_between(v1), "度")


夹角 [1,0] 与 [0,1]: 90.0 度
夹角 [1,0] 与 [1,1]: 45.0 度
夹角 [1,0] 与自己: 0.0 度


## 5. 正交投影

$$
\mathrm{proj}_{\mathbf{u}}(\mathbf{v}) = \frac{\mathbf{v}\cdot\mathbf{u}}{\mathbf{u}\cdot\mathbf{u}}\,\mathbf{u}
$$

把 $\mathbf{v}$ 拆成「沿着 $\mathbf{u}$」和「垂直于 $\mathbf{u}$」两段。残差与 $\mathbf{u}$ 点积应为 0。`[3,4]` 投到 x 轴 `[1,0]` 上，只留下 `[3, 0]`。


In [6]:
def project_onto(self, other):
    """当前向量在 other 上的正交投影。other 为零会除零。"""
    scalar = self.dot(other) / other.dot(other)
    return Vector([scalar * x for x in other.components])


Vector.project_onto = project_onto

a = Vector([3, 4])
u = Vector([1, 0])
proj = a.project_onto(u)
residual = a - proj
print("a =", a)
print("投到 [1,0] =", proj)
print("残差 =", residual)
print("残差 · [1,0] =", residual.dot(u))


a = Vector([3, 4])
投到 [1,0] = Vector([3.0, 0.0])
残差 = Vector([0.0, 4.0])
残差 · [1,0] = 0.0


## 6. 线性无关

一组向量线性无关，当且仅当没有一个能写成其余的线性组合。对行向量做高斯消元，秩等于向量个数就是无关。

标准基 $\{e_1,e_2,e_3\}$ 无关；把第三支换成 $2e_1+e_2$，秩掉到 2，相关。


In [7]:
def is_independent(vectors):
    """行简化求秩。rank == 向量个数 → 线性无关。"""
    n = len(vectors)
    if n == 0:
        return True
    dim = vectors[0].dim
    rows = [v.components[:] for v in vectors]
    rank = 0
    for col in range(dim):
        pivot = None
        for row in range(rank, len(rows)):
            if abs(rows[row][col]) > 1e-10:
                pivot = row
                break
        if pivot is None:
            continue
        rows[rank], rows[pivot] = rows[pivot], rows[rank]
        scale = rows[rank][col]
        rows[rank] = [x / scale for x in rows[rank]]
        for row in range(len(rows)):
            if row != rank and abs(rows[row][col]) > 1e-10:
                factor = rows[row][col]
                rows[row] = [
                    rows[row][j] - factor * rows[rank][j] for j in range(dim)
                ]
        rank += 1
    return rank == n


e1 = Vector([1, 0, 0])
e2 = Vector([0, 1, 0])
e3 = Vector([0, 0, 1])
dep = Vector([2, 1, 0])
print("{e1, e2, e3} 无关:", is_independent([e1, e2, e3]))
print("{e1, e2, 2e1+e2} 无关:", is_independent([e1, e2, dep]))


{e1, e2, e3} 无关: True
{e1, e2, 2e1+e2} 无关: False


## 7. Gram-Schmidt 正交化

对每个新向量，减去它在已有基上的投影，再归一化：

$$
\mathbf{w}_k = \mathbf{v}_k - \sum_{j<k}\mathrm{proj}_{\mathbf{u}_j}(\mathbf{v}_k),\quad
\mathbf{u}_k = \mathbf{w}_k / \|\mathbf{w}_k\|
$$

得到一组单位正交向量。线性相关的输入会被模长阈值丢掉。注意力、QR、PCA 都在用这套「换尺子」。


In [8]:
def gram_schmidt(vectors):
    """返回标准正交基。投影后几乎为零的向量跳过。"""
    orthonormal = []
    for v in vectors:
        w = v
        for u in orthonormal:
            w = w - w.project_onto(u)
        if w.magnitude() < 1e-10:
            continue
        orthonormal.append(w.normalize())
    return orthonormal


u1 = Vector([1, 1, 0])
u2 = Vector([1, 0, 1])
basis = gram_schmidt([u1, u2])
print("u1 =", basis[0])
print("u2 =", basis[1])
print("u1 · u2 =", round(basis[0].dot(basis[1]), 10))
print("|u1| =", round(basis[0].magnitude(), 10))
print("|u2| =", round(basis[1].magnitude(), 10))


u1 = Vector([0.7071067811865475, 0.7071067811865475, 0.0])
u2 = Vector([0.4082482904638631, -0.4082482904638631, 0.8164965809277261])
u1 · u2 = 0.0
|u1| = 1.0
|u2| = 1.0


## 5. 点积就是注意力，LoRA 是低秩补丁（学习目标）

注意力分数是 query 和 key 的点积（再缩放）。embedding 的「相近」就是大点积 / 大余弦。LoRA 不改满秩 $W$，只学两个瘦矩阵 $BA$（秩 $r\ll d$），更新是 $\Delta W=BA$，参数从 $d^2$ 变成 $2dr$。


In [9]:
q = Vector([1.0, 0.0, 0.0])
k_match = Vector([0.9, 0.1, 0.0])
k_other = Vector([0.0, 1.0, 0.0])
print("注意力分数 q·k_match =", round(q.dot(k_match), 3), "  q·k_other =", round(q.dot(k_other), 3))
d, r = 1024, 8
print(f"满秩微调参数 {d*d:,}；LoRA r={r} 参数 {2*d*r:,}  ({2*d*r/(d*d):.2%})")


注意力分数 q·k_match = 0.9   q·k_other = 0.0
满秩微调参数 1,048,576；LoRA r=8 参数 16,384  (1.56%)


## 对照表

| 函数 | 角色 |
|------|------|
| `Vector.__add__` / `__sub__` / `__mul__` | 平移箭头、拉长缩短 |
| `Vector.dot` | 相似度的原始分数 |
| `Vector.magnitude` / `normalize` | 长度；只留方向 |
| `Vector.cosine_similarity` | 夹角余弦，正交为 0 |
| `Vector.angle_between` | 夹角（度），先 clip 再 `acos` |
| `Vector.project_onto` | 沿另一支箭头的分量 |
| `is_independent` | 高斯消元：秩是否等于个数 |
| `gram_schmidt` | 换成单位正交尺子 |

要看完整打印 demo，运行：

```bash
python vectors.py
```
